# Notebook 07: Grover's Search Algorithm

---

## Overview

Grover's algorithm (1996) provides a **quadratic speedup** for unstructured search problems. Given a search space of $N = 2^n$ items, it finds a marked item using only $O(\sqrt{N})$ oracle queries, compared to $O(N)$ classically.

While the speedup is "only" quadratic (unlike the exponential speedup of Deutsch-Jozsa for a contrived problem), Grover's algorithm applies to an **enormous** class of real-world problems: database search, SAT solving, optimization, and cryptographic key search.

### What You Will Learn

1. The unstructured search problem and classical vs. quantum complexity
2. The Grover operator: Oracle (phase flip) + Diffusion (amplitude amplification)
3. Full mathematical derivation with geometric interpretation
4. Optimal number of iterations: $\lfloor \frac{\pi}{4}\sqrt{N/M} \rfloor$
5. Multiple solutions case
6. Qiskit implementation for 2, 3, and 4 qubit search with amplitude visualization

## 1. The Unstructured Search Problem

### Problem Statement

Given a Boolean function (oracle) $f: \{0,1\}^n \to \{0,1\}$ such that:

$$f(x) = \begin{cases} 1 & \text{if } x = \omega \text{ (the "winner")} \\ 0 & \text{otherwise} \end{cases}$$

Find $\omega$.

### Classical Complexity

With no structure to exploit, the best classical strategy is **random sampling** or **sequential search**:
- **Worst case:** $N = 2^n$ queries
- **Average case:** $N/2$ queries
- **Lower bound:** $\Omega(N)$ queries (information-theoretic)

### Quantum Complexity (Grover)

- **Grover's algorithm:** $O(\sqrt{N})$ queries
- **This is provably optimal** for unstructured search (BBBV theorem, 1997)

### Concrete Numbers

| $n$ (qubits) | $N = 2^n$ | Classical queries | Grover queries |
|:---:|:---:|:---:|:---:|
| 10 | 1,024 | ~512 | ~25 |
| 20 | 1,048,576 | ~524,288 | ~804 |
| 30 | ~$10^9$ | ~$5 \times 10^8$ | ~25,600 |
| 50 | ~$10^{15}$ | ~$5 \times 10^{14}$ | ~$2.5 \times 10^7$ |

For AES-128 key search: from $2^{128} \approx 3.4 \times 10^{38}$ to $2^{64} \approx 1.8 \times 10^{19}$ — this is why cryptographers recommend doubling key lengths for post-quantum security.

## 2. The Grover Operator

Grover's algorithm applies a single operator (the **Grover iterate** $G$) repeatedly to the initial uniform superposition. The operator consists of two reflections:

$$G = D \cdot O$$

where:
- $O$ is the **Oracle** (phase flip on the marked state)
- $D$ is the **Diffusion operator** (reflection about the mean)

### 2.1 The Oracle $O$

The oracle flips the phase of the marked state $|\omega\rangle$:

$$O|x\rangle = \begin{cases} -|x\rangle & \text{if } x = \omega \\ |x\rangle & \text{otherwise} \end{cases}$$

In matrix form:
$$O = I - 2|\omega\rangle\langle\omega|$$

This is a **reflection** about the hyperplane orthogonal to $|\omega\rangle$.

### 2.2 The Diffusion Operator $D$

The diffusion operator reflects the state about the uniform superposition $|s\rangle = H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{\sqrt{N}}\sum_{x=0}^{N-1}|x\rangle$:

$$D = 2|s\rangle\langle s| - I = H^{\otimes n}(2|0\rangle\langle 0| - I)H^{\otimes n}$$

Equivalently, if the current amplitudes are $\alpha_x$, and the mean amplitude is $\bar{\alpha} = \frac{1}{N}\sum_x \alpha_x$, then:

$$D: \alpha_x \mapsto 2\bar{\alpha} - \alpha_x$$

This is the "**inversion about the mean**" — amplitudes above the mean are reduced, and amplitudes below the mean are increased.

## 3. Geometric Interpretation — Amplitude Amplification

This is the most elegant way to understand Grover's algorithm.

### The 2D Subspace

Define two orthogonal states:

$$|\omega\rangle = \text{the marked state (winner)}$$

$$|\omega^\perp\rangle = \frac{1}{\sqrt{N-1}} \sum_{x \neq \omega} |x\rangle = \text{uniform superposition over non-winners}$$

The initial state $|s\rangle$ lies in the 2D plane spanned by $\{|\omega\rangle, |\omega^\perp\rangle\}$:

$$|s\rangle = \sin\theta |\omega\rangle + \cos\theta |\omega^\perp\rangle$$

where:

$$\sin\theta = \frac{1}{\sqrt{N}}, \quad \cos\theta = \sqrt{\frac{N-1}{N}}$$

For large $N$: $\theta \approx \frac{1}{\sqrt{N}} \ll 1$.

### Each Grover Iteration is a Rotation

The oracle $O$ reflects about $|\omega^\perp\rangle$, and the diffusion $D$ reflects about $|s\rangle$. The composition of two reflections is a **rotation by $2\theta$** in the $\{|\omega\rangle, |\omega^\perp\rangle\}$ plane:

$$G|\psi\rangle : \text{angle } \alpha \mapsto \alpha + 2\theta$$

After $k$ iterations:

$$G^k|s\rangle = \sin\left((2k+1)\theta\right)|\omega\rangle + \cos\left((2k+1)\theta\right)|\omega^\perp\rangle$$

### Optimal Number of Iterations

We want the angle to reach $\frac{\pi}{2}$ (so that the state aligns with $|\omega\rangle$):

$$(2k_{\text{opt}}+1)\theta = \frac{\pi}{2}$$

$$k_{\text{opt}} = \frac{\pi}{4\theta} - \frac{1}{2} \approx \frac{\pi}{4}\sqrt{N}$$

$$\boxed{k_{\text{opt}} = \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor}$$

The success probability after $k_{\text{opt}}$ iterations is:

$$P(\omega) = \sin^2\left((2k_{\text{opt}}+1)\theta\right) \geq 1 - \frac{1}{N}$$

## 4. Implementation in Qiskit

Let us build the full Grover circuit from scratch.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt

print("Imports successful!")

In [ ]:
def grover_oracle(n, marked_states):
    """
    Create a Grover oracle that flips the phase of marked states.
    
    Parameters:
    -----------
    n : int — number of qubits
    marked_states : list of str — binary strings of marked states (e.g., ['101', '110'])
    
    Returns: QuantumCircuit
    """
    oracle = QuantumCircuit(n, name='Oracle')
    
    for target in marked_states:
        # Reverse because Qiskit uses LSB ordering
        rev_target = target[::-1]
        
        # Apply X gates to qubits that should be |0⟩
        for i, bit in enumerate(rev_target):
            if bit == '0':
                oracle.x(i)
        
        # Multi-controlled Z gate: flip phase when all qubits are |1⟩
        if n == 1:
            oracle.z(0)
        elif n == 2:
            oracle.cz(0, 1)
        else:
            # Use multi-controlled Z: H on last, MCX, H on last
            oracle.h(n - 1)
            oracle.mcx(list(range(n - 1)), n - 1)
            oracle.h(n - 1)
        
        # Undo the X gates
        for i, bit in enumerate(rev_target):
            if bit == '0':
                oracle.x(i)
    
    return oracle

# Test: oracle for 3 qubits marking state |101⟩
oracle_test = grover_oracle(3, ['101'])
oracle_test.draw('mpl', style='iqp')
plt.title('Grover Oracle marking |101⟩', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def diffusion_operator(n):
    """
    Create the Grover diffusion operator (reflection about |s⟩).
    
    D = H^n (2|0⟩⟨0| - I) H^n
    
    Parameters:
    -----------
    n : int — number of qubits
    
    Returns: QuantumCircuit
    """
    diffusion = QuantumCircuit(n, name='Diffusion')
    
    # Apply H to all qubits
    diffusion.h(range(n))
    
    # Apply X to all qubits (to convert |0...0⟩ to |1...1⟩)
    diffusion.x(range(n))
    
    # Multi-controlled Z gate
    if n == 1:
        diffusion.z(0)
    elif n == 2:
        diffusion.cz(0, 1)
    else:
        diffusion.h(n - 1)
        diffusion.mcx(list(range(n - 1)), n - 1)
        diffusion.h(n - 1)
    
    # Undo X and H
    diffusion.x(range(n))
    diffusion.h(range(n))
    
    return diffusion

# Visualize diffusion for 3 qubits
diff_test = diffusion_operator(3)
diff_test.draw('mpl', style='iqp')
plt.title('Diffusion Operator (3 qubits)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def grover_circuit(n, marked_states, num_iterations=None):
    """
    Build the complete Grover circuit.
    
    Parameters:
    -----------
    n : int — number of qubits
    marked_states : list of str — binary strings of marked states
    num_iterations : int or None — number of Grover iterations (default: optimal)
    
    Returns: QuantumCircuit
    """
    N = 2**n
    M = len(marked_states)  # number of solutions
    
    # Optimal number of iterations
    if num_iterations is None:
        theta = np.arcsin(np.sqrt(M / N))
        num_iterations = max(1, int(np.round(np.pi / (4 * theta) - 0.5)))
    
    qc = QuantumCircuit(n, n)
    
    # Initialize uniform superposition
    qc.h(range(n))
    qc.barrier()
    
    # Apply Grover iterations
    oracle = grover_oracle(n, marked_states)
    diffusion = diffusion_operator(n)
    
    for i in range(num_iterations):
        qc.append(oracle, range(n))
        qc.append(diffusion, range(n))
        if i < num_iterations - 1:
            qc.barrier()
    
    qc.barrier()
    
    # Measure
    qc.measure(range(n), range(n))
    
    return qc, num_iterations

print("Grover circuit builder defined.")

## 5. Grover's Algorithm — 2-Qubit Example

With $n = 2$ qubits, $N = 4$ states. The optimal number of iterations is $\lfloor \frac{\pi}{4}\sqrt{4} \rfloor = 1$.

Let us search for the state $|11\rangle$.

In [ ]:
# 2-qubit Grover: search for |11⟩
n = 2
marked = ['11']

qc, iters = grover_circuit(n, marked)
print(f"Number of qubits: {n}")
print(f"Search space size: {2**n}")
print(f"Marked state(s): {marked}")
print(f"Optimal iterations: {iters}")

# Draw the circuit
qc.draw('mpl', style='iqp', fold=False)
plt.title(f"Grover's Algorithm (n={n}): Searching for |{''.join(marked)}⟩", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Run the simulation
sim = AerSimulator()
compiled = transpile(qc, sim)
result = sim.run(compiled, shots=2048).result()
counts = result.get_counts()

print(f"Measurement results: {counts}")
print(f"Success probability: {counts.get('11', 0) / 2048:.4f}")

plot_histogram(counts)
plt.title("Grover's Search (2 qubits): Finding |11⟩")
plt.show()

## 6. Grover's Algorithm — 3-Qubit Example

With $n = 3$ qubits, $N = 8$. Optimal iterations: $\lfloor \frac{\pi}{4}\sqrt{8} \rfloor = 2$.

In [ ]:
# 3-qubit Grover: search for |101⟩
n = 3
marked = ['101']

qc3, iters3 = grover_circuit(n, marked)
print(f"Qubits: {n}, N = {2**n}, Iterations: {iters3}")

compiled = transpile(qc3, sim)
result = sim.run(compiled, shots=2048).result()
counts = result.get_counts()

target_key = '101'
print(f"Target: |{target_key}⟩")
print(f"Success probability: {counts.get(target_key, 0) / 2048:.4f}")

plot_histogram(counts)
plt.title("Grover's Search (3 qubits): Finding |101⟩")
plt.show()

## 7. Grover's Algorithm — 4-Qubit Example

With $n = 4$ qubits, $N = 16$. Optimal iterations: $\lfloor \frac{\pi}{4}\sqrt{16} \rfloor = 3$.

In [ ]:
# 4-qubit Grover: search for |1010⟩
n = 4
marked = ['1010']

qc4, iters4 = grover_circuit(n, marked)
print(f"Qubits: {n}, N = {2**n}, Iterations: {iters4}")

compiled = transpile(qc4, sim)
result = sim.run(compiled, shots=4096).result()
counts = result.get_counts()

target_key = '1010'
print(f"Target: |{target_key}⟩")
print(f"Success probability: {counts.get(target_key, 0) / 4096:.4f}")

plot_histogram(counts)
plt.title("Grover's Search (4 qubits): Finding |1010⟩")
plt.show()

## 8. Visualizing Amplitude Evolution

Let us trace how the amplitudes evolve through each Grover iteration. This is the heart of the algorithm — watching amplitude amplification in action.

In [ ]:
def visualize_amplitude_evolution(n, marked_states, max_iters=None):
    """
    Visualize the amplitude of each basis state after each Grover iteration.
    """
    N = 2**n
    M = len(marked_states)
    theta = np.arcsin(np.sqrt(M / N))
    
    if max_iters is None:
        max_iters = int(np.ceil(np.pi / (4 * theta)))
    
    # Collect amplitudes at each iteration
    oracle = grover_oracle(n, marked_states)
    diffusion = diffusion_operator(n)
    
    all_amplitudes = []
    
    # Build circuit incrementally
    qc = QuantumCircuit(n)
    qc.h(range(n))
    
    # Record initial amplitudes
    sv = Statevector.from_instruction(qc)
    all_amplitudes.append(np.real(sv.data))
    
    for i in range(max_iters):
        qc.append(oracle, range(n))
        qc.append(diffusion, range(n))
        sv = Statevector.from_instruction(qc)
        all_amplitudes.append(np.real(sv.data))
    
    # Plot
    fig, axes = plt.subplots(1, len(all_amplitudes), figsize=(4 * len(all_amplitudes), 5))
    if len(all_amplitudes) == 1:
        axes = [axes]
    
    labels = [format(i, f'0{n}b') for i in range(N)]
    marked_indices = [int(s, 2) for s in marked_states]
    
    for idx, amps in enumerate(all_amplitudes):
        colors = ['red' if i in marked_indices else 'steelblue' for i in range(N)]
        axes[idx].bar(range(N), amps, color=colors, alpha=0.8)
        axes[idx].set_title(f'Iter {idx}' if idx > 0 else 'Initial', fontsize=11)
        axes[idx].set_ylabel('Amplitude' if idx == 0 else '')
        axes[idx].set_ylim(-0.2, 1.1)
        axes[idx].axhline(y=0, color='black', linewidth=0.5)
        axes[idx].axhline(y=1/np.sqrt(N), color='gray', linewidth=0.5, linestyle='--', label='1/√N')
        if n <= 4:
            axes[idx].set_xticks(range(N))
            axes[idx].set_xticklabels(labels, rotation=90, fontsize=7)
    
    plt.suptitle(f"Grover Amplitude Evolution (n={n}, marked={marked_states})", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Also plot the probability of the marked state vs iteration
    probs = []
    for amps in all_amplitudes:
        p = sum(amps[idx]**2 for idx in marked_indices)
        probs.append(p)
    
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.plot(range(len(probs)), probs, 'ro-', linewidth=2, markersize=10)
    ax2.set_xlabel('Number of Grover iterations', fontsize=12)
    ax2.set_ylabel('Probability of marked state', fontsize=12)
    ax2.set_title(f'Success Probability vs Iterations (n={n})', fontsize=13)
    ax2.set_xticks(range(len(probs)))
    ax2.set_ylim(0, 1.05)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='P=1')
    ax2.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

# Visualize for 3 qubits
visualize_amplitude_evolution(3, ['101'])

In [ ]:
# Visualize for 4 qubits — more iterations needed
visualize_amplitude_evolution(4, ['1010'], max_iters=5)

## 9. The Danger of Over-Iteration

An important subtlety: if you iterate **too many times**, the success probability **decreases**. The state rotates past the optimal angle and starts moving away from $|\omega\rangle$.

This is very different from classical search, where more queries never hurt.

Mathematically, after $k$ iterations:

$$P(\omega) = \sin^2\left((2k+1)\theta\right)$$

This is a **periodic function** that oscillates between 0 and 1. The first maximum occurs at $k_{\text{opt}} \approx \frac{\pi}{4}\sqrt{N}$.

In [ ]:
# Demonstrate over-iteration
def prob_vs_iterations(n, num_marked=1, max_k=None):
    """Plot theoretical success probability vs number of iterations."""
    N = 2**n
    M = num_marked
    theta = np.arcsin(np.sqrt(M / N))
    
    k_opt = int(np.round(np.pi / (4 * theta) - 0.5))
    
    if max_k is None:
        max_k = 3 * k_opt
    
    ks = np.arange(0, max_k + 1)
    probs = np.sin((2 * ks + 1) * theta)**2
    
    return ks, probs, k_opt, theta

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, n in enumerate([3, 5, 8]):
    ks, probs, k_opt, theta = prob_vs_iterations(n)
    
    axes[idx].plot(ks, probs, 'b-', linewidth=2)
    axes[idx].axvline(x=k_opt, color='red', linestyle='--', linewidth=1.5,
                      label=f'$k_{{opt}} = {k_opt}$')
    axes[idx].scatter([k_opt], [probs[k_opt]], color='red', s=100, zorder=5)
    axes[idx].set_xlabel('Iterations $k$', fontsize=12)
    axes[idx].set_ylabel('$P(\\omega)$', fontsize=12)
    axes[idx].set_title(f'$n={n}$, $N={2**n}$, $k_{{opt}}={k_opt}$', fontsize=12)
    axes[idx].legend(fontsize=10)
    axes[idx].set_ylim(-0.05, 1.1)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Success Probability vs Iterations — Over-Iteration Hurts!', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Multiple Solutions

If there are $M$ marked states out of $N$ total, Grover's algorithm still works! The geometry is the same, but the angle $\theta$ is larger:

$$\sin\theta = \sqrt{\frac{M}{N}}$$

The optimal number of iterations becomes:

$$k_{\text{opt}} = \left\lfloor \frac{\pi}{4}\sqrt{\frac{N}{M}} \right\rfloor$$

**Important implications:**
- More solutions $\Rightarrow$ fewer iterations needed
- If $M = N/4$, only 1 iteration is needed
- If $M$ is unknown, we can use the **quantum counting** algorithm to estimate $M$ first, or use exponential search

In [ ]:
# Multiple solutions: search for TWO states in 3 qubits
n = 3
marked_multi = ['010', '110']

qc_multi, iters_multi = grover_circuit(n, marked_multi)
print(f"Searching for {marked_multi} in {2**n} states")
print(f"Number of solutions M = {len(marked_multi)}")
print(f"Optimal iterations: {iters_multi}")

compiled = transpile(qc_multi, sim)
result = sim.run(compiled, shots=4096).result()
counts = result.get_counts()

total_success = sum(counts.get(s, 0) for s in marked_multi)
print(f"Success probability: {total_success / 4096:.4f}")

plot_histogram(counts)
plt.title(f"Grover's Search: Multiple Solutions {marked_multi}")
plt.show()

In [ ]:
# Visualize amplitude evolution with multiple solutions
visualize_amplitude_evolution(3, ['010', '110'], max_iters=4)

## 11. Geometric Visualization

Let us visualize the rotation in the 2D plane $\{|\omega\rangle, |\omega^\perp\rangle\}$.

In [ ]:
def plot_geometric_grover(n, M=1, num_iters=None):
    """
    Visualize Grover's algorithm as rotations in the |ω⟩, |ω⊥⟩ plane.
    """
    N = 2**n
    theta = np.arcsin(np.sqrt(M / N))
    
    if num_iters is None:
        num_iters = int(np.round(np.pi / (4 * theta) - 0.5))
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Draw unit circle
    circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', linewidth=0.5)
    ax.add_patch(circle)
    
    # Draw axes
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('$|\\omega^\\perp\\rangle$ component', fontsize=13)
    ax.set_ylabel('$|\\omega\\rangle$ component', fontsize=13)
    
    # Track the state through iterations
    colors = plt.cm.viridis(np.linspace(0, 0.9, num_iters + 1))
    
    for k in range(num_iters + 1):
        angle = (2 * k + 1) * theta
        x = np.cos(angle)  # |ω⊥⟩ component
        y = np.sin(angle)  # |ω⟩ component
        
        ax.annotate('', xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=colors[k], lw=2))
        ax.annotate(f'k={k}', xy=(x, y), fontsize=10,
                    xytext=(x + 0.05, y + 0.05))
    
    # Draw the target |ω⟩ direction
    ax.annotate('', xy=(0, 1), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1, linestyle='--'))
    ax.text(0.03, 1.02, '$|\\omega\\rangle$', fontsize=14, color='red')
    
    # Draw |ω⊥⟩
    ax.annotate('', xy=(1, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1, linestyle='--'))
    ax.text(1.02, 0.03, '$|\\omega^\\perp\\rangle$', fontsize=14, color='blue')
    
    # Show the angle θ
    arc_angles = np.linspace(0, theta, 50)
    arc_r = 0.3
    ax.plot(arc_r * np.cos(arc_angles), arc_r * np.sin(arc_angles), 'g-', linewidth=1.5)
    ax.text(arc_r * 1.2 * np.cos(theta / 2), arc_r * 1.2 * np.sin(theta / 2),
            f'$\\theta$={theta:.3f}', fontsize=12, color='green')
    
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(f'Geometric View of Grover\'s Algorithm\n'
                 f'$n={n}$, $N={N}$, $M={M}$, $\\theta={theta:.4f}$, '
                 f'$k_{{opt}}={num_iters}$', fontsize=13)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

# Plot for different n
plot_geometric_grover(3, M=1)
plot_geometric_grover(5, M=1)

## 12. Detailed Mathematical Proof

### Proof that $G^k|s\rangle = \sin((2k+1)\theta)|\omega\rangle + \cos((2k+1)\theta)|\omega^\perp\rangle$

**By induction on $k$.**

**Base case ($k=0$):** 
$$G^0|s\rangle = |s\rangle = \sin\theta|\omega\rangle + \cos\theta|\omega^\perp\rangle \quad \checkmark$$

**Inductive step:** Assume $G^k|s\rangle = \sin((2k+1)\theta)|\omega\rangle + \cos((2k+1)\theta)|\omega^\perp\rangle$.

Let $\phi_k = (2k+1)\theta$. Then $G^k|s\rangle = \sin\phi_k|\omega\rangle + \cos\phi_k|\omega^\perp\rangle$.

**Step 1: Apply Oracle $O$**

$O = I - 2|\omega\rangle\langle\omega|$ flips the $|\omega\rangle$ component:

$$O(\sin\phi_k|\omega\rangle + \cos\phi_k|\omega^\perp\rangle) = -\sin\phi_k|\omega\rangle + \cos\phi_k|\omega^\perp\rangle$$

This is a reflection about $|\omega^\perp\rangle$ — the angle changes from $\phi_k$ to $-\phi_k$ (measured from $|\omega^\perp\rangle$).

**Step 2: Apply Diffusion $D = 2|s\rangle\langle s| - I$**

$D$ reflects about $|s\rangle$, which makes angle $\theta$ with $|\omega^\perp\rangle$.

The reflection of the angle $\pi - \phi_k$ (from $|\omega^\perp\rangle$) about $\theta$ gives:

$$\text{new angle} = 2\theta - (-\phi_k) = 2\theta + \phi_k = 2\theta + (2k+1)\theta = (2(k+1)+1)\theta \quad \checkmark$$

This completes the proof. Each Grover iteration rotates the state by $2\theta$ toward $|\omega\rangle$. $\blacksquare$

## 13. Implementing the Diffusion Operator — Alternative View

Let us verify that our diffusion operator really performs "inversion about the mean" by looking at a concrete example.

In [ ]:
from qiskit.quantum_info import Operator

# Verify the diffusion operator matrix for n=2
n = 2
diff = diffusion_operator(n)
D_matrix = Operator(diff).data

print(f"Diffusion operator matrix (n={n}):")
print(np.real(D_matrix).round(4))
print()

# Expected: 2|s⟩⟨s| - I = (2/N) * ones_matrix - I
N = 2**n
expected = 2/N * np.ones((N, N)) - np.eye(N)
print(f"Expected (2|s⟩⟨s| - I):")
print(expected)
print()

print(f"Match: {np.allclose(np.real(D_matrix), expected)}")

# Demonstrate inversion about the mean
amplitudes = np.array([-0.25, 0.25, 0.25, 0.25])  # After oracle flip on |00⟩
mean = np.mean(amplitudes)
inverted = 2 * mean - amplitudes

print(f"\nExample: Inversion about the mean")
print(f"Before: {amplitudes}")
print(f"Mean:   {mean:.4f}")
print(f"After:  {inverted}")
print(f"Via matrix: {np.real(D_matrix @ amplitudes).round(4)}")

## 14. Optimality of Grover's Algorithm (BBBV Theorem)

Bennett, Bernstein, Brassard, and Vazirani (1997) proved that **no quantum algorithm** can solve unstructured search with fewer than $\Omega(\sqrt{N})$ queries. This means Grover's algorithm is **optimal**.

### Informal Argument

After $k$ oracle queries, the quantum state can differ from the "no marked item" case by at most $O(k/\sqrt{N})$ in trace distance. To distinguish the two cases reliably, we need $k = \Omega(\sqrt{N})$.

### Implications for Cryptography

For a symmetric key of $n$ bits:
- Classical brute force: $O(2^n)$ operations
- Grover search: $O(2^{n/2})$ operations
- **Solution:** Use $2n$-bit keys for $n$-bit post-quantum security
  - AES-128 $\to$ AES-256 for 128-bit quantum security

In [ ]:
# Summary comparison across qubit counts
print("Grover's Algorithm — Summary Table")
print("=" * 75)
print(f"{'n':<5} {'N=2^n':<12} {'Opt. Iters':<12} {'Classical':<15} {'Speedup':<15}")
print("-" * 75)

for n in [2, 3, 4, 5, 8, 10, 16, 20, 32, 64, 128]:
    N = 2**n
    theta = np.arcsin(1/np.sqrt(N))
    k_opt = max(1, int(np.round(np.pi / (4 * theta) - 0.5)))
    classical = N // 2
    speedup = classical / k_opt if k_opt > 0 else float('inf')
    
    print(f"{n:<5} {N:<12.4g} {k_opt:<12} {classical:<15.4g} {speedup:<15.1f}")

## 15. Exercises

1. **Multiple solutions:** Implement Grover's search for 3 marked states out of 16 (4 qubits). What is the optimal number of iterations?

2. **Unknown $M$:** Implement the exponential search strategy: run Grover with $k = 1, 2, 4, 8, \ldots$ iterations until a solution is found. Verify this still gives $O(\sqrt{N/M})$ total queries.

3. **Arbitrary oracle:** Implement a Grover oracle that searches for all $x$ such that $x > 5$ in a 3-qubit space. (Hint: this requires a comparison circuit.)

4. **Noise sensitivity:** Add noise to the Grover circuit and study how the success probability degrades with the number of iterations.

5. **Prove** that the success probability after $k$ iterations with $M$ solutions is exactly $\sin^2((2k+1)\arcsin(\sqrt{M/N}))$.

## 16. Summary

| Concept | Details |
|---------|--------|
| **Problem** | Find marked item in unstructured search space of $N$ items |
| **Classical** | $O(N)$ queries |
| **Grover** | $O(\sqrt{N})$ queries — quadratic speedup |
| **Operator** | $G = D \cdot O$ (Diffusion $\times$ Oracle) |
| **Geometry** | Rotation by $2\theta$ in $\{|\omega\rangle, |\omega^\perp\rangle\}$ plane |
| **Optimal iterations** | $k_{\text{opt}} = \lfloor \frac{\pi}{4}\sqrt{N/M} \rfloor$ |
| **Key insight** | Amplitude amplification via interference |
| **Limitation** | Over-iteration reduces success probability |

**Next:** [Notebook 08 — Shor's Factoring Algorithm](08-shors-factoring-algorithm.ipynb)